# Solutions · 18 · Markov chain Monte Carlo

Worked solutions to the exercises of [notebook 18](../notebooks/18_markov_chain_monte_carlo.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from engstoch import mcmc, output as oa
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

Sample the posterior of the logistic regression in this notebook with HMC (write the gradient) and compare effective samples per second with random-walk Metropolis.

In [2]:
rng = R(6)
x = rng.random(40) * 10
y = (rng.random(40) < 1 / (1 + np.exp(-(-3.0 + 0.8 * x)))).astype(float)
def lp(b):
    eta = b[0] + b[1] * x
    return float(np.sum(y * eta - np.logaddexp(0, eta)) - 0.5 * np.sum(b**2) / 100)
def glp(b):
    p = 1 / (1 + np.exp(-(b[0] + b[1] * x)))
    return np.array([np.sum(y - p), np.sum((y - p) * x)]) - b / 100
rows = []
for name, run in (("random-walk Metropolis", lambda: mcmc.metropolis(lp, [0, 0], 20_000, [0.6, 0.12], R(1))["chain"]),
                  ("HMC (step 0.05, 20 leapfrogs)", lambda: mcmc.hmc(lp, glp, np.zeros(2), 2000, 0.05, 20, R(2))["chain"])):
    t0 = time.perf_counter(); ch = run(); dt = time.perf_counter() - t0
    e = min(oa.ess(ch[len(ch) // 5:, j]) for j in range(2))
    rows.append((name, len(ch), e, e / dt))
print(pd.DataFrame(rows, columns=["sampler", "iterations", "min ESS", "ESS per second"]).to_string(index=False, float_format="%.1f"))

                      sampler  iterations  min ESS  ESS per second
       random-walk Metropolis       20000    141.6           551.3
HMC (step 0.05, 20 leapfrogs)        2000    279.6           451.4


HMC produces nearly independent draws per iteration but each costs twenty gradient evaluations; random-walk
Metropolis is cheap per step but strongly autocorrelated along the posterior's correlated ridge. Per second of
computing they are within a small factor here; HMC's advantage grows with dimension, where random walks slow
down linearly and HMC only like d^{1/4}.

## Exercise 2

Sample a mixture of two well-separated normals (means -4 and 4) with random-walk Metropolis from several starts. Show that R-hat detects the failure, then fix it with parallel tempering between two temperatures.

In [3]:
def logp(z):
    return float(np.logaddexp(-0.5 * (z[0] - 4) ** 2, -0.5 * (z[0] + 4) ** 2))
chains = [mcmc.metropolis(logp, [s], 20_000, 1.0, R(10 + k))["chain"][:, 0] for k, s in enumerate((-4, -4, 4, 4))]
print("single-temperature chains: means", [round(float(c.mean()), 2) for c in chains], " R-hat", round(oa.gelman_rubin([c[10_000:] for c in chains])["R_hat"], 2))
def tempering(n, g, betas=(1.0, 0.1)):
    x = np.array([4.0, 4.0]); out = np.empty(n)
    for k in range(n):
        for j, b in enumerate(betas):
            y = x[j] + 1.5 * g.standard_normal()
            if np.log(g.random()) < b * (logp([y]) - logp([x[j]])): x[j] = y
        if np.log(g.random()) < (betas[0] - betas[1]) * (logp([x[1]]) - logp([x[0]])): x = x[::-1].copy()
        out[k] = x[0]
    return out
pt = [tempering(20_000, R(20 + k)) for k in range(4)]
print("parallel tempering: means", [round(float(c.mean()), 2) for c in pt], " R-hat", round(oa.gelman_rubin([c[10_000:] for c in pt])["R_hat"], 3), " P(z > 0)", round(float(np.mean(np.concatenate(pt) > 0)), 3))

single-temperature chains: means [-0.71, -1.03, 0.25, 1.42]  R-hat 1.16


parallel tempering: means [-0.23, 0.28, 0.5, 0.04]  R-hat 1.006  P(z > 0) 0.517


Started in different modes, plain Metropolis chains rarely cross the valley between them: their means range
from −1 to 1.4 although the truth is 0, and R-hat (1.16, far above 1.01) flags the failure. A second chain at inverse temperature 0.1 sees a
flattened target and crosses freely; swapping states between the two (accepted with the ratio of the tempered
densities) carries the crossings to the cold chain, which now spends half its time in each mode.

## Exercise 3

**Implement it yourself:** write the checkerboard Metropolis sweep for the Ising model from scratch and reproduce the energy per site at beta = 0.3 against `mcmc.onsager_energy`.

In [4]:
L, beta, sweeps = 32, 0.3, 2000
g = R(30); s = g.choice(np.array([-1, 1]), (L, L))
ii, jj = np.indices((L, L)); masks = [(ii + jj) % 2 == 0, (ii + jj) % 2 == 1]
E = []
for k in range(sweeps):
    for mask in masks:
        nb = np.roll(s, 1, 0) + np.roll(s, -1, 0) + np.roll(s, 1, 1) + np.roll(s, -1, 1)
        dE = 2 * s * nb
        flip = mask & (g.random((L, L)) < np.exp(-beta * np.maximum(dE, 0)))
        s = np.where(flip, -s, s)
    E.append(-np.sum(s * (np.roll(s, 1, 0) + np.roll(s, 1, 1))) / L**2)
E = np.array(E[300:])
print(f"energy per site {E.mean():.4f} +- {E.std() * np.sqrt(oa.iat(E)['tau'] / len(E)):.4f}; Onsager {mcmc.onsager_energy(beta):.4f}")

energy per site -0.7049 +- 0.0018; Onsager -0.7045


Sites of one checkerboard colour have neighbours only of the other colour, so all of them can be updated at
once without violating detailed balance; a sweep is two vectorised half-sweeps. At β = 0.3 (high temperature)
the 32 × 32 lattice already reproduces the infinite-lattice energy within its error bar.